In [1]:
# Gradio eklendi
!pip install -q -U transformers peft accelerate bitsandbytes gradio

In [2]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 22 packages in 3s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [ ]:
import torch
from google.colab import drive
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Drive'ı Bağla (Eğer bağlı değilse onay isteyecektir)
drive.mount('/content/drive')

# 2. Kurulum ve Yükleme
base_model_name = "mistralai/Mistral-7B-v0.1"
adapter_path = "/content/drive/MyDrive/mistral_final_model_altın/mistral-7b-turkce-duzeltici"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Model Drive'dan yükleniyor...")
base_model = AutoModelForCausalLM.from_pretrained(base_model_name, quantization_config=bnb_config, device_map="auto")
model_to_test = PeftModel.from_pretrained(base_model, adapter_path) # Değişken adı: model_to_test
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token
model_to_test.eval()

# 3. Senin Fonksiyonun (İsim hatası düzeltildi)
def paraphrase_final(metin):
    prompt = (
        f"### Instruction:\n"
        f"Sen profesyonel bir Türkçe paraphrase modülüsün. Girdi metnindeki anlamı kesinlikle koruyarak, "
        f"daha zengin, akıcı ve kurallı bir Türkçe ile yeniden ifade et. Metne asla yeni bilgi ekleme.\n\n"
        f"### Input:\n{metin}\n\n"
        f"### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        # Burada model_to_test kullandığımıza emin oluyoruz
        outputs = model_to_test.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Response:" in result:
        return result.split("### Response:")[1].strip().split("###")[0].strip()
    return result

# 4. Final Testi
bozuk_cumle = "Savaş zamanı sorunları fix etmek zor olur."
print("\n" + "="*30)
print(f"GİRDİ: {bozuk_cumle}")
print(f"ÇIKTI: {paraphrase_final(bozuk_cumle)}")
print("="*30)

Mounted at /content/drive
Model Drive'dan yükleniyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 7bd8cfa9-627b-4d2d-a730-235e86844609)')' thrown while requesting HEAD https://huggingface.co/mistralai/Mistral-7B-v0.1/resolve/main/special_tokens_map.json
Retrying in 1s [Retry 1/5].


special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



GİRDİ: Savaş zamanı sorunları fix etmek zor olur.
ÇIKTI: Kuantum teknolojilerini kullanarak savaş zamanındaki operasyonel sorunlara yönelik giderek gecikmeyi minimize ederek, operasyonel verimliliği artırmak mümkündür.


In [3]:
import gradio as gr
import torch
import time
from google.colab import drive
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. MODELİ VE YOLLARI TANIMLA
drive.mount('/content/drive')

base_model_name = "mistralai/Mistral-7B-v0.1"
adapter_path = "/content/drive/MyDrive/mistral_final_model_altın/mistral-7b-turkce-duzeltici"

# 2. MODELİ YÜKLE (4-bit Kuantizasyon)
print("📦 Model yükleme süreci başladı, lütfen bekleyin...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model_to_test = PeftModel.from_pretrained(base_model, adapter_path)
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token
model_to_test.eval()

print("✅ Model ve Adapter başarıyla yüklendi!")

# 3. OPTİMİZE EDİLMİŞ DÜZELTİCİ FONKSİYON
def duzeltici_bot(input_text):
    if not input_text.strip():
        return "Lütfen düzeltilmesini istediğiniz bir metin girin."

    # Eğitim formatına sadık prompt
    prompt = (
        f"### Instruction:\n"
        f"Sen profesyonel bir Türkçe paraphrase modülüsün. Girdi metnindeki anlamı kesinlikle koruyarak, "
        f"daha zengin, akıcı ve kurallı bir Türkçe ile yeniden ifade et. Metne asla yeni bilgi ekleme.\n\n"
        f"### Input:\n{input_text}\n\n"
        f"### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[1] # Girdi uzunluğunu kaydet

    # no_grad yerine daha hızlı olan inference_mode kullanıyoruz
    with torch.inference_mode():
        outputs = model_to_test.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.3, # Daha tutarlı ve ciddi cevaplar
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    # Sadece modelin ürettiği YENİ tokenları al (Girdiyi decode etmeye vakit harcamaz)
    new_tokens = outputs[0][input_length:]
    decoded_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Temizlik: Eğer model fazladan ### etiketi üretirse temizle
    final_result = decoded_output.split("###")[0].strip()
    return final_result

# 4. MODERN GRADIO ARAYÜZÜ
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛡️ Pro-Turkce AI Redaktör (Final)")
    gr.Markdown("Daha profesyonel, zengin ve kurallı bir Türkçe için tasarlandı.")

    with gr.Row():
        with gr.Column():
            input_box = gr.Textbox(
                label="Bozuk veya Basit Metin",
                placeholder="Düzeltilmesini istediğiniz cümleyi buraya yazın...",
                lines=5
            )
            submit_btn = gr.Button("✨ Profesyonel Dile Dönüştür", variant="primary")

            gr.Examples(
                examples=[
                    ["System crash oldu because server çok hot oldu."],
                    ["şarjı bitti arabanın lastiğinin."],
                    ["Sistem dün bozuldu biz hiçbir şey yapamadık admin gelince düzeldi."]
                ],
                inputs=input_box
            )

        with gr.Column():
            output_box = gr.Textbox(label="Zenginleştirilmiş Sonuç", lines=8, interactive=False)

    # Buton aksiyonu
    submit_btn.click(fn=duzeltici_bot, inputs=input_box, outputs=output_box)

# 5. KUYRUK SİSTEMİYLE BAŞLAT (504 Hatalarını Önler)
print("🚀 Arayüz başlatılıyor...")

# (Kodunun en sonu)
demo.queue().launch(share=False, inline=True, server_name="0.0.0.0", server_port=7860)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 Model yükleme süreci başladı, lütfen bekleyin...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 24d46167-12bf-48e5-a610-48997abcda9d)')' thrown while requesting HEAD https://huggingface.co/mistralai/Mistral-7B-v0.1/resolve/main/custom_generate/generate.py
Retrying in 1s [Retry 1/5].


✅ Model ve Adapter başarıyla yüklendi!


/tmp/ipython-input-4086820818.py:75: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


🚀 Arayüz başlatılıyor...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [ ]:
# Colab'ın IP adresini öğrenelim (Tünele giriş şifren olacak)
import urllib
print("Şifren (Endpoint IP):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

# Tüneli 7860 portu üzerinden internete aç
!npx localtunnel --port 7860

Şifren (Endpoint IP): 34.142.171.229
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧your url is: https://fruity-hats-love.loca.lt


In [3]:
def duzeltici_bot(input_text):
    if not input_text.strip(): return "Lütfen bir metin girin."

    prompt = (
        f"### Instruction:\nSen profesyonel bir Türkçe paraphrase modülüsün. Anlamı koruyarak zenginleştir.\n\n"
        f"### Input:\n{input_text}\n\n"
        f"### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # inference_mode daha hızlıdır
    with torch.inference_mode():
        outputs = model_to_test.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    # Sadece yeni üretilen kısmı alıyoruz (Input ID uzunluğunu atlayarak)
    decoded_output = tokenizer.decode(outputs[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

    # Eğer model yine de ### Response yazarsa temizle
    final_output = decoded_output.split("###")[0].strip()
    return final_output

# Launch kısmına bir timeout süresi ekleyelim (Opsiyonel)
demo.queue().launch(share=True, inline=True, debug=True)

NameError: name 'demo' is not defined